# WorldSim v3 — Data Generation, Assembly & Dual-Model Training
English-only logic + new tasks (O~T) + 0.8B & 2B fine-tuning


## 1. Environment & Prerequisites

In [1]:
from pathlib import Path
import json
import os
import sys

from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "config" / "generation.yaml").exists():
        raise RuntimeError("Cannot find repo root. Run this notebook from the worldsim-training directory.")

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from scripts.common import read_jsonl

load_dotenv(REPO_ROOT / ".env")
assert os.environ.get("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY not set"

v2_train = REPO_ROOT / "data" / "final" / "worldsim-v2-mix" / "train.jsonl"
v2_dev = REPO_ROOT / "data" / "final" / "worldsim-v2-mix" / "dev.jsonl"
assert v2_train.exists(), f"v2 train not found: {v2_train}"
assert v2_dev.exists(), f"v2 dev not found: {v2_dev}"

v2_train_count = sum(1 for line in open(v2_train, encoding="utf-8") if line.strip())
v2_dev_count = sum(1 for line in open(v2_dev, encoding="utf-8") if line.strip())
v2_total_count = v2_train_count + v2_dev_count

batch1_path = REPO_ROOT / "config" / "batches" / "batch_v3_01_english_logic.yaml"
batch2_path = REPO_ROOT / "config" / "batches" / "batch_v3_02_new_tasks.yaml"
assert batch1_path.exists(), f"Batch 1 config not found: {batch1_path}"
assert batch2_path.exists(), f"Batch 2 config not found: {batch2_path}"

LLAMA_QUANTIZE = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-quantize"
LLAMA_COMPLETION = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-completion"
CONVERT_SCRIPT = REPO_ROOT / "tools" / "llama.cpp" / "convert_hf_to_gguf.py"
GGUF_DIR = REPO_ROOT / "artifacts" / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:        {REPO_ROOT}")
print("API key:          ✅ set")
print(f"v2 train data:    {v2_train} ({v2_train_count} rows)")
print(f"v2 dev data:      {v2_dev} ({v2_dev_count} rows)")
print(f"Batch 1 config:   {batch1_path.name}")
print(f"Batch 2 config:   {batch2_path.name}")
print(f"llama-quantize:   {'✅' if LLAMA_QUANTIZE.exists() else '❌'}")
print(f"llama-completion: {'✅' if LLAMA_COMPLETION.exists() else '❌'}")
print(f"convert script:   {'✅' if CONVERT_SCRIPT.exists() else '❌'}")
print(f"GGUF output:      {GGUF_DIR}")
print("✅ Environment ready")


Repo root:        /home/hyunlord/github/worldsim-training
API key:          ✅ set
v2 train data:    /home/hyunlord/github/worldsim-training/data/final/worldsim-v2-mix/train.jsonl (3455 rows)
v2 dev data:      /home/hyunlord/github/worldsim-training/data/final/worldsim-v2-mix/dev.jsonl (366 rows)
Batch 1 config:   batch_v3_01_english_logic.yaml
Batch 2 config:   batch_v3_02_new_tasks.yaml
llama-quantize:   ✅
llama-completion: ✅
convert script:   ✅
GGUF output:      /home/hyunlord/github/worldsim-training/artifacts/gguf
✅ Environment ready


## 2. Batch Preview (Dry Run)

In [2]:
from collections import Counter

from scripts.generate_data import (
    _build_jobs_from_catalogs,
    apply_batch_plan_to_settings,
    batch_task_counts,
    load_batch_plan,
    load_catalogs,
    load_generation_config,
)

settings = load_generation_config(REPO_ROOT / "config")
catalogs = load_catalogs(REPO_ROOT / "config")

for batch_name in ["batch_v3_01_english_logic", "batch_v3_02_new_tasks"]:
    plan = load_batch_plan(REPO_ROOT, batch_id=batch_name)
    merged = apply_batch_plan_to_settings(settings, plan)
    schema_version = int(merged.get("schema_version", 2))
    target_counts = batch_task_counts(plan) or {}
    all_jobs = _build_jobs_from_catalogs(catalogs, merged, schema_version=schema_version)
    available_counts = Counter(job["task"] for job in all_jobs)

    print()
    print("=" * 60)
    print(f"  {batch_name} (schema_version={schema_version})")
    print("=" * 60)
    for task, target in sorted(target_counts.items()):
        available = available_counts.get(task, 0)
        print(f"  Task {task}: target={target} available={available}")
    print(f"  Total requested: {sum(target_counts.values())}")

print()
print("✅ Dry run complete")



  batch_v3_01_english_logic (schema_version=3)
  Task E: target=200 available=200
  Task F: target=250 available=800
  Task H: target=50 available=50
  Task I: target=200 available=200
  Task J: target=200 available=200
  Task K: target=200 available=200
  Task L: target=200 available=200
  Task M: target=200 available=240
  Task N: target=200 available=240
  Total requested: 1700

  batch_v3_02_new_tasks (schema_version=3)
  Task O: target=500 available=1500
  Task P: target=500 available=1500
  Task Q: target=500 available=1500
  Task R: target=400 available=1500
  Task S: target=400 available=1500
  Task T: target=400 available=1500
  Total requested: 2700

✅ Dry run complete


## 3. Batch 1: English Logic Tasks (E~N)

In [3]:
import time

from scripts.generate_data import generate_dataset, load_batch_plan

batch_plan_1 = load_batch_plan(REPO_ROOT, batch_id="batch_v3_01_english_logic")
target_1 = sum((batch_plan_1.get("task_counts") or batch_plan_1.get("tasks", {})).values())
result_1 = None
batch1_error = None
elapsed_1 = None

print(f"Starting Batch 1: English Logic E~N ({target_1} requests)", flush=True)
print("Estimated cost: ~$6", flush=True)
print("Estimated time: 30-60 minutes", flush=True)
print("=" * 50, flush=True)

started_1 = time.perf_counter()
try:
    result_1 = generate_dataset(REPO_ROOT, batch_plan=batch_plan_1)
except Exception as exc:  # noqa: BLE001
    batch1_error = exc
    print(f"Batch 1 failed: {type(exc).__name__}: {exc}", flush=True)
else:
    elapsed_1 = time.perf_counter() - started_1
    print()
    print("=" * 50, flush=True)
    print(f"Batch 1 complete in {elapsed_1:.0f}s ({elapsed_1 / 60:.1f}min)", flush=True)

result_1


Starting Batch 1: English Logic E~N (1700 requests)
Estimated cost: ~$6
Estimated time: 30-60 minutes
Batch 1 failed: FileExistsError: Refusing to overwrite existing generation artifact: /home/hyunlord/github/worldsim-training/data/raw/batch_v3_01_english_logic/generated.jsonl


## 4. Batch 1 Summary

In [4]:
from collections import Counter

from scripts.common import read_jsonl

batch1_dir = REPO_ROOT / "data" / "raw" / "batch_v3_01_english_logic"
batch1_generated = batch1_dir / "generated.jsonl"
batch1_skipped = batch1_dir / "skipped.jsonl"
batch1_summary_path = batch1_dir / "summary.json"
batch1_summary = json.loads(batch1_summary_path.read_text(encoding="utf-8")) if batch1_summary_path.exists() else {}

if batch1_summary:
    print("=== Batch 1 Summary ===")
    print(f"  successful_rows: {batch1_summary.get('successful_rows', 'N/A')}")
    print(f"  skipped_rows:    {batch1_summary.get('skipped_rows', 'N/A')}")
    print(f"  success_rate:    {batch1_summary.get('success_rate', 'N/A')}")
    usage = batch1_summary.get("usage", {})
    print(f"  estimated_cost:  ${usage.get('estimated_cost_usd', 0.0):.4f}")
    print(f"  total_tokens:    {usage.get('total_tokens', 0)}")

generated_rows_1 = read_jsonl(batch1_generated) if batch1_generated.exists() else []
skipped_rows_1 = read_jsonl(batch1_skipped) if batch1_skipped.exists() else []
print()
print(f"Generated rows: {len(generated_rows_1)}")
print(f"Skipped rows:   {len(skipped_rows_1)}")
print("Per-task generated:")
for task, count in sorted(Counter(row.get("task", "?") for row in generated_rows_1).items()):
    print(f"  Task {task}: {count}")


=== Batch 1 Summary ===
  successful_rows: 1492
  skipped_rows:    208
  success_rate:    0.877647
  estimated_cost:  $9.7032
  total_tokens:    2516376

Generated rows: 1492
Skipped rows:   208
Per-task generated:
  Task F: 250
  Task H: 50
  Task I: 200
  Task J: 200
  Task K: 200
  Task L: 200
  Task M: 200
  Task N: 192


## 5. Batch 2: New Tasks (O~T)

In [5]:
batch_plan_2 = load_batch_plan(REPO_ROOT, batch_id="batch_v3_02_new_tasks")
target_2 = sum((batch_plan_2.get("task_counts") or batch_plan_2.get("tasks", {})).values())
result_2 = None
batch2_error = None
elapsed_2 = None

print(f"Starting Batch 2: New Tasks O~T ({target_2} requests)", flush=True)
print("Estimated cost: ~$11", flush=True)
print("Estimated time: 60-120 minutes", flush=True)
print("=" * 50, flush=True)

started_2 = time.perf_counter()
try:
    result_2 = generate_dataset(REPO_ROOT, batch_plan=batch_plan_2)
except Exception as exc:  # noqa: BLE001
    batch2_error = exc
    print(f"Batch 2 failed: {type(exc).__name__}: {exc}", flush=True)
else:
    elapsed_2 = time.perf_counter() - started_2
    print()
    print("=" * 50, flush=True)
    print(f"Batch 2 complete in {elapsed_2:.0f}s ({elapsed_2 / 60:.1f}min)", flush=True)

result_2


Starting Batch 2: New Tasks O~T (2700 requests)
Estimated cost: ~$11
Estimated time: 60-120 minutes
Batch 2 failed: FileExistsError: Refusing to overwrite existing generation artifact: /home/hyunlord/github/worldsim-training/data/raw/batch_v3_02_new_tasks/generated.jsonl


## 6. Batch 2 Summary

In [6]:
from collections import Counter

batch2_dir = REPO_ROOT / "data" / "raw" / "batch_v3_02_new_tasks"
batch2_generated = batch2_dir / "generated.jsonl"
batch2_skipped = batch2_dir / "skipped.jsonl"
batch2_summary_path = batch2_dir / "summary.json"
batch2_summary = json.loads(batch2_summary_path.read_text(encoding="utf-8")) if batch2_summary_path.exists() else {}

if batch2_summary:
    print("=== Batch 2 Summary ===")
    print(f"  successful_rows: {batch2_summary.get('successful_rows', 'N/A')}")
    print(f"  skipped_rows:    {batch2_summary.get('skipped_rows', 'N/A')}")
    print(f"  success_rate:    {batch2_summary.get('success_rate', 'N/A')}")
    usage = batch2_summary.get("usage", {})
    print(f"  estimated_cost:  ${usage.get('estimated_cost_usd', 0.0):.4f}")
    print(f"  total_tokens:    {usage.get('total_tokens', 0)}")

generated_rows_2 = read_jsonl(batch2_generated) if batch2_generated.exists() else []
skipped_rows_2 = read_jsonl(batch2_skipped) if batch2_skipped.exists() else []
print()
print(f"Generated rows: {len(generated_rows_2)}")
print(f"Skipped rows:   {len(skipped_rows_2)}")
print("Per-task generated:")
for task, count in sorted(Counter(row.get("task", "?") for row in generated_rows_2).items()):
    print(f"  Task {task}: {count}")


=== Batch 2 Summary ===
  successful_rows: 2661
  skipped_rows:    39
  success_rate:    0.985556
  estimated_cost:  $13.9781
  total_tokens:    3404985

Generated rows: 2661
Skipped rows:   39
Per-task generated:
  Task O: 480
  Task P: 500
  Task Q: 491
  Task R: 390
  Task S: 400
  Task T: 400


## 7. Validation

In [7]:
from scripts.common import load_yaml
from scripts.validate_data import _load_batch_plan as load_validation_batch_plan
from scripts.validate_data import _resolve_batch_paths, load_validation_rules, validate_file

validation_settings = load_yaml(REPO_ROOT / "config" / "generation.yaml")
validation_rules = load_validation_rules(REPO_ROOT / "config")
validation_reports = {}

for batch_id in ["batch_v3_01_english_logic", "batch_v3_02_new_tasks"]:
    print()
    print(f"Validating {batch_id}...", flush=True)
    batch_plan = load_validation_batch_plan(REPO_ROOT, batch_id)
    input_path, output_dir = _resolve_batch_paths(REPO_ROOT, validation_settings, batch_plan)
    if not input_path.exists():
        print(f"  ⏭️ missing raw file: {input_path}")
        continue
    report = validate_file(input_path=input_path, validated_dir=output_dir, rules=validation_rules)
    validation_reports[batch_id] = report
    print(f"  Passed: {report.get('passed', 0)}")
    print(f"  Failed: {report.get('failed', 0)}")
    print(f"  Output: {output_dir}")



Validating batch_v3_01_english_logic...
  Passed: 1492
  Failed: 0
  Output: /home/hyunlord/github/worldsim-training/data/validated/batch_v3_01_english_logic

Validating batch_v3_02_new_tasks...
  Passed: 2661
  Failed: 0
  Output: /home/hyunlord/github/worldsim-training/data/validated/batch_v3_02_new_tasks


## 8. Assemble v3 Dataset

In [8]:
from scripts.assemble_v3_dataset import assemble_v3_dataset

V3_DATASET_ID = "worldsim-v3-mix"
assembly_result = assemble_v3_dataset(
    REPO_ROOT,
    dataset_id=V3_DATASET_ID,
    dev_ratio=0.1,
    seed=42,
)
assembly_manifest = assembly_result.manifest
v3_final_dir = REPO_ROOT / "data" / "final" / V3_DATASET_ID

print()
print("=" * 60)
print("  V3 DATASET ASSEMBLED")
print("=" * 60)
print(f"  Train: {assembly_manifest['output']['train']}")
print(f"  Dev:   {assembly_manifest['output']['dev']}")
print(f"  Total: {assembly_manifest['output']['total']}")
print(f"  Dedup removed: {assembly_manifest['deduplication']['removed']}")
print(f"  Output: {v3_final_dir}")
assembly_manifest



  V3 DATASET ASSEMBLED
  Train: 5856
  Dev:   645
  Total: 6501
  Dedup removed: 622
  Output: /home/hyunlord/github/worldsim-training/data/final/worldsim-v3-mix


{'dataset_id': 'worldsim-v3-mix',
 'generated_at': '2026-03-17T03:40:59.445582+00:00',
 'dev_ratio': 0.1,
 'seed': 42,
 'sources': {'v2_train_filtered': 1939,
  'v2_dev_filtered': 231,
  'logic_total': 1492,
  'new_tasks_total': 2661,
  'negatives': 500,
  'general': 300},
 'deduplication': {'removed': 622},
 'output': {'train': 5856,
  'dev': 645,
  'train_curriculum': 5856,
  'total': 6501},
 'task_counts': {'train': {'A': 88,
   'B': 800,
   'C': 389,
   'F': 225,
   'G': 662,
   'GEN': 30,
   'H': 45,
   'I': 180,
   'J': 180,
   'K': 180,
   'L': 179,
   'M': 180,
   'N': 173,
   'NEG': 158,
   'O': 430,
   'P': 449,
   'Q': 442,
   'R': 346,
   'S': 360,
   'T': 360},
  'dev': {'A': 12,
   'B': 96,
   'C': 40,
   'F': 25,
   'G': 83,
   'H': 5,
   'I': 19,
   'J': 20,
   'K': 20,
   'L': 20,
   'M': 20,
   'N': 19,
   'O': 48,
   'P': 50,
   'Q': 49,
   'R': 39,
   'S': 40,
   'T': 40},
  'train_curriculum': {'A': 88,
   'B': 800,
   'C': 389,
   'F': 225,
   'G': 662,
   'GEN': 

## 9. v3 Dataset Stats

In [9]:
from collections import Counter

from scripts.curriculum_order_v3 import CURRICULUM_STAGES_V3

v3_train = read_jsonl(v3_final_dir / "train.jsonl")
v3_dev = read_jsonl(v3_final_dir / "dev.jsonl")
v3_curriculum = read_jsonl(v3_final_dir / "train_curriculum.jsonl")

print("=" * 60)
print("  V3 DATASET BREAKDOWN")
print("=" * 60)
print()
print(f"  Train: {len(v3_train)}, Dev: {len(v3_dev)}, Curriculum: {len(v3_curriculum)}")

print()
print("  Task distribution (train):")
task_counts = Counter(row.get("task", "?") for row in v3_train)
for task, count in sorted(task_counts.items()):
    lang = "KO" if task in {"A", "B", "C", "G"} else "EN"
    bar = "█" * max(1, count // 20)
    print(f"    {task:>3}: {count:>5} [{lang}] {bar}")
print(f"  Total: {sum(task_counts.values())}")

print()
print("  v2 → v3 comparison:")
print(f"    v2: {v2_total_count} rows, 15 tasks, bilingual")
print(f"    v3: {len(v3_train) + len(v3_dev)} rows, 20 tasks, EN logic + KO narration")

print()
print("  Curriculum stages:")
for stage, tasks in sorted(CURRICULUM_STAGES_V3.items()):
    stage_count = sum(1 for row in v3_curriculum if row.get("task") in tasks)
    print(f"    Stage {stage}: {sorted(tasks)} → {stage_count} rows ({stage_count / len(v3_curriculum) * 100:.1f}%)")


  V3 DATASET BREAKDOWN

  Train: 5856, Dev: 645, Curriculum: 5856

  Task distribution (train):
      A:    88 [KO] ████
      B:   800 [KO] ████████████████████████████████████████
      C:   389 [KO] ███████████████████
      F:   225 [EN] ███████████
      G:   662 [KO] █████████████████████████████████
    GEN:    30 [EN] █
      H:    45 [EN] ██
      I:   180 [EN] █████████
      J:   180 [EN] █████████
      K:   180 [EN] █████████
      L:   179 [EN] ████████
      M:   180 [EN] █████████
      N:   173 [EN] ████████
    NEG:   158 [EN] ███████
      O:   430 [EN] █████████████████████
      P:   449 [EN] ██████████████████████
      Q:   442 [EN] ██████████████████████
      R:   346 [EN] █████████████████
      S:   360 [EN] ██████████████████
      T:   360 [EN] ██████████████████
  Total: 5856

  v2 → v3 comparison:
    v2: 3821 rows, 15 tasks, bilingual
    v3: 6501 rows, 20 tasks, EN logic + KO narration

  Curriculum stages:
    Stage 1: ['E', 'F', 'I', 'J'] → 585 rows (

## 10. Convert to Training Format

In [10]:
import math

from scripts.common import write_jsonl
from scripts.convert_mixed_final_to_training_format import convert_mixed_final_to_training_format
from scripts.curriculum_order_v3 import curriculum_order_v3
from training.lib.qlora_smoke import (
    SmokeRunConfig,
    load_json_artifact,
    load_optional_json_artifact,
    load_sample_generations,
    run_baseline_or_raise,
    summarize_sample_generations,
)

V3_TRAINING_DIR = REPO_ROOT / "data" / "training" / V3_DATASET_ID

conversion_result = convert_mixed_final_to_training_format(
    repo_root=REPO_ROOT,
    input_train=v3_final_dir / "train.jsonl",
    input_dev=v3_final_dir / "dev.jsonl",
    source_manifest=assembly_result.manifest_path,
    output_dir=V3_TRAINING_DIR,
    dataset_id=V3_DATASET_ID,
)

train_converted = conversion_result.train_output
dev_converted = conversion_result.dev_output
train_curriculum = V3_TRAINING_DIR / "train_curriculum.jsonl"
converted_train_rows = read_jsonl(train_converted)
ordered_train_rows = curriculum_order_v3(converted_train_rows, seed=42)
write_jsonl(train_curriculum, ordered_train_rows)

train_converted_count = len(converted_train_rows)
dev_converted_count = conversion_result.dev_count
train_curriculum_count = len(ordered_train_rows)


def compute_three_epoch_steps(row_count: int, *, batch_size: int = 1, grad_accum: int = 8, epochs: int = 3) -> int:
    effective_batch = max(1, batch_size * grad_accum)
    steps_per_epoch = max(1, math.ceil(row_count / effective_batch))
    return steps_per_epoch * epochs


def load_guardrail_bundle(output_dir: str | Path) -> tuple[dict, dict, dict, dict | None]:
    run_dir = Path(output_dir)
    summary_artifact = load_json_artifact(run_dir, "summary.json")
    metrics_artifact = load_optional_json_artifact(run_dir, "metrics.json") or {}
    sample_rows = load_sample_generations(run_dir)
    sample_summary = summarize_sample_generations(sample_rows) if sample_rows else None
    structured_metrics = metrics_artifact.get("structured_metrics", {})
    return summary_artifact, metrics_artifact, structured_metrics, sample_summary


max_steps_v3 = compute_three_epoch_steps(train_curriculum_count, batch_size=1, grad_accum=8, epochs=3)

print(f"Training format output: {V3_TRAINING_DIR}")
print(f"  train_converted:   {train_converted_count} rows")
print(f"  dev_converted:     {dev_converted_count} rows")
print(f"  train_curriculum:  {train_curriculum_count} rows")
print(f"  computed max_steps (3 epochs): {max_steps_v3}")
print("✅ Conversion complete")


Training format output: /home/hyunlord/github/worldsim-training/data/training/worldsim-v3-mix
  train_converted:   5856 rows
  dev_converted:     645 rows
  train_curriculum:  5856 rows
  computed max_steps (3 epochs): 2196
✅ Conversion complete


## 11. Train 0.8B on v3 Data

In [11]:
import time
from datetime import UTC, datetime

MODEL_08B = "Qwen/Qwen3.5-0.8B-Base"
RUN_ID_08B = datetime.now(UTC).strftime("run-%Y%m%dT%H%M%SZ")
OUTPUT_08B = REPO_ROOT / "outputs" / "baseline" / "worldsim-v3-08b" / RUN_ID_08B

cfg_08b = SmokeRunConfig(
    run_mode="baseline",
    model_name=MODEL_08B,
    train_file=str(train_curriculum),
    dev_file=str(dev_converted),
    output_dir=str(OUTPUT_08B),
    max_steps=max_steps_v3,
    max_train_samples=0,
    max_eval_samples=0,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=10,
    eval_steps=100,
    save_steps=100,
    save_total_limit=1,
    require_qlora=True,
    seed=42,
    dry_run=False,
)

result_08b = None
summary_artifact_08b = {}
structured_metrics_08b = {}
sample_summary_08b = None
error_08b = None
elapsed_08b = None

print("Training 0.8B on v3 data...", flush=True)
print(f"  Model:  {MODEL_08B}", flush=True)
print(f"  Data:   {train_curriculum}", flush=True)
print(f"  Output: {OUTPUT_08B}", flush=True)

started_08b = time.perf_counter()
try:
    result_08b = run_baseline_or_raise(cfg_08b)
    summary_artifact_08b, metrics_artifact_08b, structured_metrics_08b, sample_summary_08b = load_guardrail_bundle(result_08b.output_dir)
except Exception as exc:  # noqa: BLE001
    error_08b = exc
    print(f"0.8B training failed: {type(exc).__name__}: {exc}", flush=True)
else:
    elapsed_08b = time.perf_counter() - started_08b
    print()
    print("=" * 60)
    print(f"  0.8B TRAINING COMPLETE ({elapsed_08b / 60:.1f} min)")
    print("=" * 60)
    for key in ["train_loss", "eval_loss", "output_dir", "adapter_dir"]:
        print(f"  {key}: {summary_artifact_08b.get(key, 'N/A')}")
    if structured_metrics_08b:
        for key in ["per_sample_success_rate", "structured_success_rate", "json_parse_failure_rate", "repair_applied_rate"]:
            value = structured_metrics_08b.get(key, "N/A")
            print(f"  {key}: {f'{value:.4f}' if isinstance(value, float) else value}")

result_08b


Training 0.8B on v3 data...
  Model:  Qwen/Qwen3.5-0.8B-Base
  Data:   /home/hyunlord/github/worldsim-training/data/training/worldsim-v3-mix/train_curriculum.jsonl
  Output: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v3-08b/run-20260317T034101Z


/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights:   2%|▏         | 6/320 [00:02<06:52,  1.31s/it, Materializing param=model.layers.0.linear_attn.in_proj_a.weight]/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 320/320 [00:07<00:00, 4

Step,Training Loss,Validation Loss
100,0.248803,0.246389
200,0.175712,0.151946
300,0.127206,0.124683
400,0.106608,0.110792
500,0.104586,0.102205
600,0.120132,0.096263
700,0.103349,0.092299
800,0.086831,0.089323
900,0.102336,0.087045
1000,0.083515,0.085482


/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1258: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1258: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between

[outlines] Constrained decoding ENABLED (model wrapped successfully)


/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



  0.8B TRAINING COMPLETE (205.8 min)
  train_loss: 0.12205663723752364
  eval_loss: 0.07473950833082199
  output_dir: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v3-08b/run-20260317T034101Z
  adapter_dir: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v3-08b/run-20260317T034101Z/adapter
  per_sample_success_rate: 1.0000
  structured_success_rate: 1.0000
  json_parse_failure_rate: 0.0000
  repair_applied_rate: 0.0000


SmokeRunResult(success=True, status='ok', used_true_qlora=True, runtime={'device': 'cuda', 'use_qlora': True, 'fallback_reason': None, 'torch_dtype': 'bfloat16'}, environment={'python': {'version': '3.12.3', 'executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3'}, 'platform': {'system': 'Linux', 'release': '6.14.0-1015-nvidia', 'machine': 'aarch64'}, 'cwd': '/home/hyunlord/github/worldsim-training', 'torch': {'available': True, 'version': '2.12.0.dev20260225+cu130', 'cuda_version': '13.0', 'cuda_available': True, 'mps_available': False, 'cuda_device_count': 1, 'cuda_device_name': 'NVIDIA GB10', 'cuda_device_names': ['NVIDIA GB10'], 'cuda_bf16_supported': True}, 'transformers': {'available': True, 'version': '5.2.0'}, 'datasets': {'available': True, 'version': '4.7.0'}, 'peft': {'available': True, 'version': '0.18.1'}, 'trl': {'available': False, 'error': "ModuleNotFoundError: No module named 'trl'"}, 'accelerate': {'available': True, 'version': '1.12.0'}, 'bitsandbytes

## 12. Train 2B on v3 Data

In [12]:
MODEL_2B = "Qwen/Qwen3.5-2B-Base"
RUN_ID_2B = datetime.now(UTC).strftime("run-%Y%m%dT%H%M%SZ")
OUTPUT_2B = REPO_ROOT / "outputs" / "baseline" / "worldsim-v3-2b" / RUN_ID_2B

cfg_2b = SmokeRunConfig(
    run_mode="baseline",
    model_name=MODEL_2B,
    train_file=str(train_curriculum),
    dev_file=str(dev_converted),
    output_dir=str(OUTPUT_2B),
    max_steps=max_steps_v3,
    max_train_samples=0,
    max_eval_samples=0,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=10,
    eval_steps=100,
    save_steps=100,
    save_total_limit=1,
    require_qlora=True,
    seed=42,
    dry_run=False,
)

result_2b = None
summary_artifact_2b = {}
structured_metrics_2b = {}
sample_summary_2b = None
error_2b = None
elapsed_2b = None

print("Training 2B on v3 data...", flush=True)
print(f"  Model:  {MODEL_2B}", flush=True)
print(f"  Data:   {train_curriculum}", flush=True)
print(f"  Output: {OUTPUT_2B}", flush=True)

started_2b = time.perf_counter()
try:
    result_2b = run_baseline_or_raise(cfg_2b)
    summary_artifact_2b, metrics_artifact_2b, structured_metrics_2b, sample_summary_2b = load_guardrail_bundle(result_2b.output_dir)
except Exception as exc:  # noqa: BLE001
    error_2b = exc
    print(f"2B training failed: {type(exc).__name__}: {exc}", flush=True)
else:
    elapsed_2b = time.perf_counter() - started_2b
    print()
    print("=" * 60)
    print(f"  2B TRAINING COMPLETE ({elapsed_2b / 60:.1f} min)")
    print("=" * 60)
    for key in ["train_loss", "eval_loss", "output_dir", "adapter_dir"]:
        print(f"  {key}: {summary_artifact_2b.get(key, 'N/A')}")
    if structured_metrics_2b:
        for key in ["per_sample_success_rate", "structured_success_rate", "json_parse_failure_rate", "repair_applied_rate"]:
            value = structured_metrics_2b.get(key, "N/A")
            print(f"  {key}: {f'{value:.4f}' if isinstance(value, float) else value}")

result_2b


Training 2B on v3 data...
  Model:  Qwen/Qwen3.5-2B-Base
  Data:   /home/hyunlord/github/worldsim-training/data/training/worldsim-v3-mix/train_curriculum.jsonl
  Output: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v3-2b/run-20260317T070651Z


Loading weights:   2%|▏         | 6/320 [00:05<15:31,  2.97s/it, Materializing param=model.layers.0.linear_attn.in_proj_a.weight]/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 320/320 [00:17<00:00, 18.08it/s, Materializing param=model.norm.weight]                              
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.
/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1258: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. S

Step,Training Loss,Validation Loss
100,0.181659,0.178695
200,0.133343,0.116702
300,0.103750,0.099232
400,0.088604,0.090047
500,0.087418,0.084965
600,0.098948,0.080970
700,0.088816,0.079052
800,0.071397,0.076713
900,0.085663,0.075440
1000,0.071916,0.074465


/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1258: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1258: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between

[outlines] Constrained decoding ENABLED (model wrapped successfully)


/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



  2B TRAINING COMPLETE (270.9 min)
  train_loss: 0.10083947827418645
  eval_loss: 0.0662321001291275
  output_dir: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v3-2b/run-20260317T070651Z
  adapter_dir: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v3-2b/run-20260317T070651Z/adapter
  per_sample_success_rate: 1.0000
  structured_success_rate: 1.0000
  json_parse_failure_rate: 0.0000
  repair_applied_rate: 0.0000


SmokeRunResult(success=True, status='ok', used_true_qlora=True, runtime={'device': 'cuda', 'use_qlora': True, 'fallback_reason': None, 'torch_dtype': 'bfloat16'}, environment={'python': {'version': '3.12.3', 'executable': '/home/hyunlord/github/diffusion-study/.venv/bin/python3'}, 'platform': {'system': 'Linux', 'release': '6.14.0-1015-nvidia', 'machine': 'aarch64'}, 'cwd': '/home/hyunlord/github/worldsim-training', 'torch': {'available': True, 'version': '2.12.0.dev20260225+cu130', 'cuda_version': '13.0', 'cuda_available': True, 'mps_available': False, 'cuda_device_count': 1, 'cuda_device_name': 'NVIDIA GB10', 'cuda_device_names': ['NVIDIA GB10'], 'cuda_bf16_supported': True}, 'transformers': {'available': True, 'version': '5.2.0'}, 'datasets': {'available': True, 'version': '4.7.0'}, 'peft': {'available': True, 'version': '0.18.1'}, 'trl': {'available': False, 'error': "ModuleNotFoundError: No module named 'trl'"}, 'accelerate': {'available': True, 'version': '1.12.0'}, 'bitsandbytes

## 13. GGUF Conversion — Both Models

In [13]:
import shutil
import subprocess

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


def merge_and_convert_gguf(model_name: str, output_dir: Path, gguf_name: str) -> Path:
    adapter_dir = output_dir / "adapter"
    merged_dir = output_dir / "merged_bf16"
    fp16_gguf = output_dir / f"{gguf_name}-f16.gguf"
    q4_gguf = output_dir / f"{gguf_name}-q4_k_m.gguf"

    if not (merged_dir / "config.json").exists():
        print(f"  Merging LoRA for {model_name}...", flush=True)
        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=False,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
        merged_model = PeftModel.from_pretrained(base_model, str(adapter_dir)).merge_and_unload()
        merged_dir.mkdir(parents=True, exist_ok=True)
        merged_model.save_pretrained(str(merged_dir))
        tokenizer.save_pretrained(str(merged_dir))
        del merged_model, base_model
        torch.cuda.empty_cache()
        print("  Merged ✅", flush=True)
    else:
        print("  Already merged ✅", flush=True)

    if not fp16_gguf.exists():
        print("  Converting to GGUF FP16...", flush=True)
        conversion = subprocess.run(
            [sys.executable, str(CONVERT_SCRIPT), str(merged_dir), "--outfile", str(fp16_gguf), "--outtype", "f16"],
            capture_output=True,
            text=True,
        )
        if conversion.returncode != 0:
            raise RuntimeError(f"GGUF conversion failed: {conversion.stderr[-500:]}")
        print(f"  FP16 GGUF: {fp16_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)
    else:
        print("  FP16 GGUF exists ✅", flush=True)

    if not q4_gguf.exists():
        print("  Quantizing to Q4_K_M...", flush=True)
        quant = subprocess.run(
            [str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), "Q4_K_M"],
            capture_output=True,
            text=True,
        )
        if quant.returncode != 0:
            raise RuntimeError(f"Quantization failed: {quant.stderr[-500:]}")
        print(f"  Q4_K_M GGUF: {q4_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)
    else:
        print("  Q4_K_M GGUF exists ✅", flush=True)

    return q4_gguf


gguf_08b = None
gguf_2b = None
final_08b = GGUF_DIR / "worldsim-v3-qwen3.5-0.8b-q4_k_m.gguf"
final_2b = GGUF_DIR / "worldsim-v3-qwen3.5-2b-q4_k_m.gguf"

if result_08b is not None:
    print("=== 0.8B GGUF Conversion ===", flush=True)
    gguf_08b = merge_and_convert_gguf(MODEL_08B, Path(result_08b.output_dir), "worldsim-v3-qwen3.5-0.8b")
    shutil.copy2(gguf_08b, final_08b)
else:
    print("⏭️ Skipping 0.8B GGUF conversion (training did not complete)")

print()
if result_2b is not None:
    print("=== 2B GGUF Conversion ===", flush=True)
    gguf_2b = merge_and_convert_gguf(MODEL_2B, Path(result_2b.output_dir), "worldsim-v3-qwen3.5-2b")
    shutil.copy2(gguf_2b, final_2b)
else:
    print("⏭️ Skipping 2B GGUF conversion (training did not complete)")

print()
print("=" * 60)
print("  DUAL MODEL GGUF COMPLETE")
print("=" * 60)
if final_08b.exists():
    print(f"  0.8B: {final_08b} ({final_08b.stat().st_size / 1e6:.0f} MB)")
else:
    print("  0.8B: not produced")
if final_2b.exists():
    print(f"  2B:   {final_2b} ({final_2b.stat().st_size / 1e6:.0f} MB)")
else:
    print("  2B:   not produced")
if final_08b.exists() and final_2b.exists():
    combined_mb = (final_08b.stat().st_size + final_2b.stat().st_size) / 1e6
    print(f"  Combined: {combined_mb:.0f} MB")


=== 0.8B GGUF Conversion ===
  Merging LoRA for Qwen/Qwen3.5-0.8B-Base...


`torch_dtype` is deprecated! Use `dtype` instead!
Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.27s/it]


  Merged ✅
  Converting to GGUF FP16...
  FP16 GGUF: 1517 MB ✅
  Quantizing to Q4_K_M...
  Q4_K_M GGUF: 529 MB ✅

=== 2B GGUF Conversion ===
  Merging LoRA for Qwen/Qwen3.5-2B-Base...


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.04s/it]

  Merged ✅
  Converting to GGUF FP16...


  FP16 GGUF: 3776 MB ✅
  Quantizing to Q4_K_M...
  Q4_K_M GGUF: 1274 MB ✅

  DUAL MODEL GGUF COMPLETE
  0.8B: /home/hyunlord/github/worldsim-training/artifacts/gguf/worldsim-v3-qwen3.5-0.8b-q4_k_m.gguf (529 MB)
  2B:   /home/hyunlord/github/worldsim-training/artifacts/gguf/worldsim-v3-qwen3.5-2b-q4_k_m.gguf (1274 MB)
  Combined: 1804 MB


## 14. Grand Summary

In [14]:
batch1_cost = batch1_summary.get("usage", {}).get("estimated_cost_usd", 0.0) if isinstance(batch1_summary, dict) else 0.0
batch2_cost = batch2_summary.get("usage", {}).get("estimated_cost_usd", 0.0) if isinstance(batch2_summary, dict) else 0.0

print("=" * 70)
print("  WORLDSIM V3 — COMPLETE PIPELINE RESULTS")
print("=" * 70)

print()
print("  [Data Generation]")
print(f"    Batch 1 (English E~N): {f'{elapsed_1 / 60:.1f} min' if elapsed_1 is not None else 'not completed'}")
print(f"    Batch 2 (New O~T):     {f'{elapsed_2 / 60:.1f} min' if elapsed_2 is not None else 'not completed'}")
print(f"    Total API cost:        ~${batch1_cost + batch2_cost:.2f}")

print()
print("  [Dataset]")
print(f"    v3 train:     {len(v3_train)} rows")
print(f"    v3 dev:       {len(v3_dev)} rows")
print("    Tasks:        20 (A~T)")
print("    Languages:    EN logic + KO narration")

print()
print("  [Training]")
print(f"    0.8B train_loss: {summary_artifact_08b.get('train_loss', 'N/A')}")
print(f"    0.8B eval_loss:  {summary_artifact_08b.get('eval_loss', 'N/A')}")
print(f"    2B train_loss:   {summary_artifact_2b.get('train_loss', 'N/A')}")
print(f"    2B eval_loss:    {summary_artifact_2b.get('eval_loss', 'N/A')}")
print(f"    0.8B per-sample success: {structured_metrics_08b.get('per_sample_success_rate', 'N/A')}")
print(f"    2B per-sample success:   {structured_metrics_2b.get('per_sample_success_rate', 'N/A')}")

print()
print("  [GGUF Models]")
if final_08b.exists():
    print(f"    0.8B: {final_08b.name} ({final_08b.stat().st_size / 1e6:.0f} MB)")
else:
    print("    0.8B: not produced")
if final_2b.exists():
    print(f"    2B:   {final_2b.name} ({final_2b.stat().st_size / 1e6:.0f} MB)")
else:
    print("    2B:   not produced")

print()
print("  [v2 → v3 Comparison]")
print(f"    v2: {v2_total_count} rows, 15 tasks, bilingual, 0.8B only")
print(f"    v3: {len(v3_train) + len(v3_dev)} rows, 20 tasks, EN+KO split, dual model")
print("=" * 70)


  WORLDSIM V3 — COMPLETE PIPELINE RESULTS

  [Data Generation]
    Batch 1 (English E~N): not completed
    Batch 2 (New O~T):     not completed
    Total API cost:        ~$23.68

  [Dataset]
    v3 train:     5856 rows
    v3 dev:       645 rows
    Tasks:        20 (A~T)
    Languages:    EN logic + KO narration

  [Training]
    0.8B train_loss: 0.12205663723752364
    0.8B eval_loss:  0.07473950833082199
    2B train_loss:   0.10083947827418645
    2B eval_loss:    0.0662321001291275
    0.8B per-sample success: 1.0
    2B per-sample success:   1.0

  [GGUF Models]
    0.8B: worldsim-v3-qwen3.5-0.8b-q4_k_m.gguf (529 MB)
    2B:   worldsim-v3-qwen3.5-2b-q4_k_m.gguf (1274 MB)

  [v2 → v3 Comparison]
    v2: 3821 rows, 15 tasks, bilingual, 0.8B only
    v3: 6501 rows, 20 tasks, EN+KO split, dual model
